# STEP1 - 데이터 로딩 및 검증

1주차 전처리 결과물(`전처리 데이터/train.csv`, `val.csv`)을 그대로 로드하고,
건수(29,200/3,640)·라벨 컬럼 순서·결측치를 검증한다. 새 전처리는 하지 않는다.

산출물: `train_results/train_val_split_check.txt`

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import common

print("PROJECT_ROOT:", common.PROJECT_ROOT)
print("TRAIN_CSV:", common.TRAIN_CSV)
print("VAL_CSV:", common.VAL_CSV)

PROJECT_ROOT: C:\2026 데이터+AI 크리에이터 캠프
TRAIN_CSV: C:\2026 데이터+AI 크리에이터 캠프\전처리 데이터\train.csv
VAL_CSV: C:\2026 데이터+AI 크리에이터 캠프\전처리 데이터\val.csv


In [2]:
train_df = common.load_split_csv(common.TRAIN_CSV)
val_df = common.load_split_csv(common.VAL_CSV)

print("train shape:", train_df.shape)
print("val shape:", val_df.shape)
train_df.head(3)

train shape: (29200, 10)
val shape: (3640, 10)


,text,고열,구토,두통,복통,어지러움,열상,오심,전신쇠약,호흡곤란
0,"119대원: 예 119입니다.\n신고자: 예, 어제 코로나 백신 주사를 맞고요. 너...",0,1,1,1,1,0,0,0,0
1,"119대원: 119입니다. 말씀하세요.\n신고자: 네, 안녕하세요, 저 지금 오른쪽...",0,1,0,0,0,0,0,0,0
2,"119대원: 119입니다. 무슨 일이세요?\n신고자: 아, 네, 여보세요.\n119...",0,0,1,0,0,0,0,0,0


In [3]:
EXPECTED_TRAIN = 29200
EXPECTED_VAL = 3640

count_lines = []
count_lines.append(f"train 건수: {len(train_df)} (기대값 {EXPECTED_TRAIN}) -> {'일치' if len(train_df) == EXPECTED_TRAIN else '불일치'}")
count_lines.append(f"val 건수: {len(val_df)} (기대값 {EXPECTED_VAL}) -> {'일치' if len(val_df) == EXPECTED_VAL else '불일치'}")
count_lines.append(f"라벨 컬럼 순서: {list(train_df.columns[1:])}")
count_lines.append(f"기대 순서(common.TARGET_SYMPTOMS): {common.TARGET_SYMPTOMS}")
count_lines.append(f"순서 일치: {list(train_df.columns[1:]) == common.TARGET_SYMPTOMS}")

for line in count_lines:
    print(line)

train 건수: 29200 (기대값 29200) -> 일치
val 건수: 3640 (기대값 3640) -> 일치
라벨 컬럼 순서: ['고열', '구토', '두통', '복통', '어지러움', '열상', '오심', '전신쇠약', '호흡곤란']
기대 순서(common.TARGET_SYMPTOMS): ['고열', '구토', '두통', '복통', '어지러움', '열상', '오심', '전신쇠약', '호흡곤란']
순서 일치: True


In [4]:
label_cols = common.TARGET_SYMPTOMS

def check_labels(df, split_name):
    msgs = []
    na_count = int(df[label_cols].isna().sum().sum())
    msgs.append(f"[{split_name}] 라벨 결측 셀 수: {na_count}")

    unique_vals = sorted(int(v) for v in pd.unique(df[label_cols].values.ravel()))
    msgs.append(f"[{split_name}] 라벨 고유값: {unique_vals} (0/1만 있어야 정상)")

    text_na = int(df['text'].isna().sum())
    msgs.append(f"[{split_name}] text 결측 수: {text_na}")
    return msgs

train_msgs = check_labels(train_df, "train")
val_msgs = check_labels(val_df, "val")

for m in train_msgs + val_msgs:
    print(m)

[train] 라벨 결측 셀 수: 0
[train] 라벨 고유값: [0, 1] (0/1만 있어야 정상)
[train] text 결측 수: 0
[val] 라벨 결측 셀 수: 0
[val] 라벨 고유값: [0, 1] (0/1만 있어야 정상)
[val] text 결측 수: 0


In [5]:
train_counts = train_df[label_cols].sum().rename("train_positive_count")
val_counts = val_df[label_cols].sum().rename("val_positive_count")
dist_df = pd.concat([train_counts, val_counts], axis=1)
dist_df["train_ratio"] = (dist_df["train_positive_count"] / len(train_df)).round(4)
dist_df["val_ratio"] = (dist_df["val_positive_count"] / len(val_df)).round(4)
dist_df

,train_positive_count,val_positive_count,train_ratio,val_ratio
고열,5186,678,0.1776,0.1863
구토,4463,537,0.1528,0.1475
두통,2905,374,0.0995,0.1027
복통,6771,838,0.2319,0.2302
어지러움,6334,774,0.2169,0.2126
열상,3465,463,0.1187,0.1272
오심,3341,429,0.1144,0.1179
전신쇠약,5310,651,0.1818,0.1788
호흡곤란,3927,492,0.1345,0.1352


In [8]:
check_lines = []
check_lines.append("=== STEP1: train/val 데이터 검증 로그 ===")
check_lines.append("")
check_lines.extend(count_lines)
check_lines.append("")
check_lines.append("[결측/값 검증]")
check_lines.extend(train_msgs)
check_lines.extend(val_msgs)
check_lines.append("")
check_lines.append("[증상별 양성 건수/비율]")
check_lines.append(dist_df.to_string())

with open(common.SPLIT_CHECK_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(check_lines))

print(f"저장 완료 -> {common.SPLIT_CHECK_PATH}")
print("\n".join(check_lines))

저장 완료 -> C:\2026 데이터+AI 크리에이터 캠프\3주차\train_results\train_val_split_check.txt
=== STEP1: train/val 데이터 검증 로그 ===

train 건수: 29200 (기대값 29200) -> 일치
val 건수: 3640 (기대값 3640) -> 일치
라벨 컬럼 순서: ['고열', '구토', '두통', '복통', '어지러움', '열상', '오심', '전신쇠약', '호흡곤란']
기대 순서(common.TARGET_SYMPTOMS): ['고열', '구토', '두통', '복통', '어지러움', '열상', '오심', '전신쇠약', '호흡곤란']
순서 일치: True

[결측/값 검증]
[train] 라벨 결측 셀 수: 0
[train] 라벨 고유값: [0, 1] (0/1만 있어야 정상)
[train] text 결측 수: 0
[val] 라벨 결측 셀 수: 0
[val] 라벨 고유값: [0, 1] (0/1만 있어야 정상)
[val] text 결측 수: 0

[증상별 양성 건수/비율]
      train_positive_count  val_positive_count  train_ratio  val_ratio
고열                    5186                 678       0.1776     0.1863
구토                    4463                 537       0.1528     0.1475
두통                    2905                 374       0.0995     0.1027
복통                    6771                 838       0.2319     0.2302
어지러움                  6334                 774       0.2169     0.2126
열상                    3465                 